# **Actividad #1**
* Paula Barillas - 22764
* Gerardo Pineda - 22880
* Monica Salvatierra - 22249
* Bianca Calderón - 22272
* Francis Aguilar - 22243
* José Marchena - 22398

Link del repositorio: https://github.com/paulabaal12/ACT1-ML

# Diseño de un Pipeline de Scikit-learn con `transactions.csv`

Este notebook utiliza el dataset `transactions.csv`. El pipeline incluirá etapas de preparación de datos, filtrado, manejo de diferentes tipos de variables, y la separación del dataset para un modelo de machine learning. También abordaremos la visualización del pipeline, la discusión sobre su empaquetado para compartir con el equipo y su ejecución en diferentes computadoras, todo ello haciendo referencia a las primeras dos etapas de CRISP-DM.

In [90]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn import set_config

print("Librerías importadas exitosamente.")

Librerías importadas exitosamente.


## 1. Etapas CRISP-DM: Business Understanding y Data Understanding

Las dos primeras etapas del ciclo de vida de CRISP-DM (Cross-Industry Standard Process for Data Mining) son `Business Understanding` y `Data Understanding`. A continuación, aplicaremos estos conceptos al contexto del dataset `transactions.csv` y el diseño de nuestro pipeline.

### 2. Data Understanding (Comprensión de los Datos)

En esta etapa, exploraremos el dataset `transactions.csv` para familiarizarnos con su estructura, identificar tipos de datos, detectar valores ausentes y entender las características disponibles, lo cual es fundamental para el diseño de las etapas de preprocesamiento del pipeline.

In [91]:
# Cargar el dataset transactions.csv
df = pd.read_csv('/content/transactions.csv')

print("Primeras 5 filas del dataset transactions.csv:")
display(df.head())

print("\nInformación del dataset transactions.csv:")
df.info()

print("\nValores nulos por columna:")
display(df.isnull().sum())

Primeras 5 filas del dataset transactions.csv:


,Transaction ID,Timestamp,Sender Name,Sender UPI ID,Receiver Name,Receiver UPI ID,Amount (INR),Status
0,4d3db980-46cd-4158-a812-dcb77055d0d2,2024-06-22 04:06:38,Tiya Mall,4161803452@okaxis,Mohanlal Golla,7776849307@okybl,3907.34,FAILED
1,099ee548-2fc1-4811-bf92-559c467ca792,2024-06-19 06:04:49,Mohanlal Bakshi,8908837379@okaxis,Mehul Sankaran,7683454560@okaxis,8404.55,SUCCESS
2,d4c05732-6b1b-4bab-90b9-efe09d252b99,2024-06-04 04:56:09,Kismat Bora,4633654150@okybl,Diya Goel,2598130823@okicici,941.88,SUCCESS
3,e8df92ee-8b04-4133-af5a-5f412180c8ab,2024-06-09 09:56:07,Ayesha Korpal,7018842771@okhdfcbank,Rhea Kothari,2246623650@okaxis,8926.00,SUCCESS
4,e7d675d3-04f1-419c-a841-7a04662560b7,2024-06-25 08:38:19,Jivin Batta,1977143985@okybl,Baiju Issac,5245672729@okybl,2800.55,SUCCESS



Información del dataset transactions.csv:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 8 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Transaction ID   1000 non-null   object 
 1   Timestamp        1000 non-null   object 
 2   Sender Name      1000 non-null   object 
 3   Sender UPI ID    1000 non-null   object 
 4   Receiver Name    1000 non-null   object 
 5   Receiver UPI ID  1000 non-null   object 
 6   Amount (INR)     1000 non-null   float64
 7   Status           1000 non-null   object 
dtypes: float64(1), object(7)
memory usage: 62.6+ KB

Valores nulos por columna:


,0
Transaction ID,0
Timestamp,0
Sender Name,0
Sender UPI ID,0
Receiver Name,0
Receiver UPI ID,0
Amount (INR),0
Status,0


### 3. Preparación del Dataset `transactions.csv` para el Pipeline

En esta sección, realizaremos las siguientes operaciones de preparación de datos:

*   **Extracción y Filtrado:** Utilizaremos todas las columnas relevantes del dataset. Eliminaremos cualquier columna redundante si existe.
*   **Manejo de Tipos de Variables:** Crearemos una variable objetivo binaria, `is_large_transaction`, a partir de la columna 'Amount (INR)' (si existe), binarizándola en base a la mediana. Esto convertirá el problema en una tarea de clasificación.
*   **Identificación de Características:** Determinaremos explícitamente qué columnas son numéricas y cuáles son categóricas para el preprocesamiento del pipeline.

In [92]:
# Crear una copia del dataframe original para trabajar con ella
df_procesado = df.copy()

# Crear la variable objetivo 'is_large_transaction' basada en 'Amount (INR)'
# Asumimos que 'Amount (INR)' es la columna numérica para el target.
if 'Amount (INR)' in df_procesado.columns:
    median_amount = df_procesado['Amount (INR)'].median()
    df_procesado['is_large_transaction'] = (df_procesado['Amount (INR)'] > median_amount).astype(int)
    target_column = 'is_large_transaction'
    # Columnas a excluir de las características (features) X
    cols_to_exclude_from_X = ['Amount (INR)', target_column]
else:
    print("Advertencia: La columna 'Amount (INR)' no se encontró. Creando un target dummy.")
    df_procesado['is_large_transaction'] = np.random.randint(0, 2, len(df_procesado))
    target_column = 'is_large_transaction'
    cols_to_exclude_from_X = [target_column]

# Identificar columnas numéricas y categóricas para el pipeline
# Excluir la columna original del 'Amount' y la columna target del conjunto de características

caracteristicas_numericas = df_procesado.select_dtypes(include=np.number).columns.tolist()
# Eliminar el target_column y la columna original 'Amount (INR)' si están en las numéricas
for col_to_remove in cols_to_exclude_from_X:
    if col_to_remove in caracteristicas_numericas:
        caracteristicas_numericas.remove(col_to_remove)

caracteristicas_categoricas = df_procesado.select_dtypes(include='object').columns.tolist()

# Separar características (X) y objetivo (y)
X = df_procesado.drop(columns=cols_to_exclude_from_X, errors='ignore')
y = df_procesado[target_column]

print(f"Variable objetivo creada: '{target_column}'")
print(f"Columnas numéricas identificadas para el pipeline: {caracteristicas_numericas}")
print(f"Columnas categóricas identificadas para el pipeline: {caracteristicas_categoricas}")

print("\nPrimeras 5 filas de X (características):")
display(X.head())
print("\nPrimeras 5 filas de y (variable objetivo):")
display(y.head())

Variable objetivo creada: 'is_large_transaction'
Columnas numéricas identificadas para el pipeline: []
Columnas categóricas identificadas para el pipeline: ['Transaction ID', 'Timestamp', 'Sender Name', 'Sender UPI ID', 'Receiver Name', 'Receiver UPI ID', 'Status']

Primeras 5 filas de X (características):


,Transaction ID,Timestamp,Sender Name,Sender UPI ID,Receiver Name,Receiver UPI ID,Status
0,4d3db980-46cd-4158-a812-dcb77055d0d2,2024-06-22 04:06:38,Tiya Mall,4161803452@okaxis,Mohanlal Golla,7776849307@okybl,FAILED
1,099ee548-2fc1-4811-bf92-559c467ca792,2024-06-19 06:04:49,Mohanlal Bakshi,8908837379@okaxis,Mehul Sankaran,7683454560@okaxis,SUCCESS
2,d4c05732-6b1b-4bab-90b9-efe09d252b99,2024-06-04 04:56:09,Kismat Bora,4633654150@okybl,Diya Goel,2598130823@okicici,SUCCESS
3,e8df92ee-8b04-4133-af5a-5f412180c8ab,2024-06-09 09:56:07,Ayesha Korpal,7018842771@okhdfcbank,Rhea Kothari,2246623650@okaxis,SUCCESS
4,e7d675d3-04f1-419c-a841-7a04662560b7,2024-06-25 08:38:19,Jivin Batta,1977143985@okybl,Baiju Issac,5245672729@okybl,SUCCESS



Primeras 5 filas de y (variable objetivo):


,is_large_transaction
0,0
1,1
2,0
3,1
4,0


### 4. Diseño de la Etapa de Preprocesamiento

Definiremos los transformadores específicos para las variables numéricas y categóricas. Para las numéricas, utilizaremos `SimpleImputer` con la estrategia de la media para manejar valores nulos y `StandardScaler` para escalar. Para las categóricas, usaremos `SimpleImputer` con la estrategia de la moda (most_frequent) y `OneHotEncoder` para convertirlas en un formato numérico.

In [93]:
# Crear preprocesadores para cada tipo de columna
transformador_numerico = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

transformador_categorico = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# Combinar los preprocesadores usando ColumnTransformer
preprocesador = ColumnTransformer(
    transformers=[
        ('num', transformador_numerico, caracteristicas_numericas),
        ('cat', transformador_categorico, caracteristicas_categoricas)
    ])

print("Preprocesadores definidos.")

Preprocesadores definidos.


### 5. Creación del Pipeline Completo

Ahora, construiremos el pipeline completo que encapsula la etapa de preprocesamiento y un modelo de Machine Learning. Para este ejemplo, utilizaremos una `LogisticRegression`.

In [94]:
# Definir el pipeline completo
full_pipeline = Pipeline(steps=[
    ('preprocesador', preprocesador),
    ('clasificador', LogisticRegression(solver='liblinear', max_iter=1000)) # Aumentar max_iter por si acaso
])

print("Pipeline completo definido.")

Pipeline completo definido.


### 6. Separación del Dataset (Entrenamiento y Prueba)

Antes de entrenar el pipeline, es fundamental dividir el dataset en conjuntos de entrenamiento y prueba para evaluar el rendimiento del modelo en datos no vistos.

In [95]:
# Dividir el dataset en conjuntos de entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Tamaño del conjunto de entrenamiento: {X_train.shape}")
print(f"Tamaño del conjunto de prueba: {X_test.shape}")

Tamaño del conjunto de entrenamiento: (800, 7)
Tamaño del conjunto de prueba: (200, 7)


### 7. Visualización del Diagrama del Pipeline

Scikit-learn ofrece una excelente funcionalidad para visualizar la estructura del pipeline, lo cual es muy útil para comprender la secuencia de transformaciones y el modelo final.

In [96]:
# diagrama del pipeline
set_config(display='diagram')
display(full_pipeline)

# representación textual
set_config(display='text')
print("\nRepresentación textual del pipeline:\n")
print(full_pipeline)

Pipeline(steps=[('preprocesador',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer()),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  []),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['Transaction ID',
                                                   'Timestamp', 'Sender Name',
                                                   'Sender UPI ID',
                                                   'Receiver Name',
                                                   'Receiver UPI ID',
                                                   'Status'])])),
                ('clasificador',
                 LogisticRegression(max_iter=1000, solver='liblinear'))])


Representación textual del pipeline:

Pipeline(steps=[('preprocesador',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer()),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  []),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['Tran

### 8. Entrenamiento y Evaluación del Pipeline

Con el pipeline completamente definido y el dataset dividido, procederemos a entrenar el modelo utilizando el conjunto de entrenamiento y a evaluar su rendimiento en el conjunto de prueba.

In [97]:
# Entrenar el pipeline
print("Entrenando el pipeline...")
full_pipeline.fit(X_train, y_train)
print("Pipeline entrenado exitosamente.")

# Realizar predicciones en el conjunto de prueba
y_pred = full_pipeline.predict(X_test)

# Evaluar el rendimiento del modelo
accuracy = accuracy_score(y_test, y_pred)
report = classification_report(y_test, y_pred)

print(f"\nPrecisión del modelo en el conjunto de prueba: {accuracy:.2f}")
print("\nReporte de Clasificación:\n")
print(report)

Entrenando el pipeline...
Pipeline entrenado exitosamente.

Precisión del modelo en el conjunto de prueba: 0.44

Reporte de Clasificación:

              precision    recall  f1-score   support

           0       0.44      0.98      0.61        90
           1       0.00      0.00      0.00       110

    accuracy                           0.44       200
   macro avg       0.22      0.49      0.31       200
weighted avg       0.20      0.44      0.28       200



### 9. Empaquetado del Pipeline para Compartir con Compañeros

creamos un paquete dentro del workspace para que el pipeline pueda reutilizarse desde otro script. Separamos la lógica de preprocesamiento y modelado en un módulo independiente, para que otros compañeros puedan usarlo sin reescribir el código.

En esta parte se genera:
- un paquete Python llamado `mi_proyecto_pipeline`,
- un archivo de instalación con `setup.py`,
- una pequeña guía de uso y un script de prueba.


In [ ]:
import pandas as pd
from pathlib import Path
import sys

sys.path.append(str(Path.cwd()))
from mi_proyecto_pipeline.pipeline import entrenar_y_evaluar

# Cargar el archivo de ejemplo y ejecutar el pipeline desde el paquete
ruta_datos = Path("transactions_demo.csv")
df = pd.read_csv(ruta_datos)
pipeline, accuracy, report = entrenar_y_evaluar(df, target_column="is_large_transaction")

print("El paquete se importó correctamente y el pipeline se ejecutó.")
print(f"Precisión obtenida: {accuracy:.2f}")
print("\nReporte de clasificación:\n")
print(report)


### 10. Demostración de Ejecución en Diferentes Computadoras

Creamos un script reutilizable que importe el pipeline empaquetado y lo ejecute con los mismos datos desde otro archivo. Esto simula lo que haría un compañero en otra computadora: instalar el paquete o agregar el directorio al entorno y ejecutar el código.

A continuación se crea una estructura simple de proyecto y se ejecuta la demostración desde un script externo.


In [ ]:
from pathlib import Path
import pandas as pd
import sys

sys.path.append(str(Path.cwd()))
from mi_proyecto_pipeline.pipeline import entrenar_y_evaluar

# Simulación de una ejecución en otra computadora
ruta_datos = Path("transactions_demo.csv")
df = pd.read_csv(ruta_datos)
_, accuracy, report = entrenar_y_evaluar(df, target_column="is_large_transaction")

print("Ejecución demostrativa desde un entorno externo")
print(f"Resultado reproducible: precisión = {accuracy:.2f}")
print("\nEste ejemplo muestra cómo un compañero podría ejecutar el mismo pipeline en otra máquina con los mismos datos.")
